 Import thư viện

In [1]:
import pandas as pd
import numpy as np
import json
import gzip
import os
import sys
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Thêm thư mục gốc vào path để import config
ROOT_DIR = Path().resolve().parent
sys.path.append(str(ROOT_DIR))

print(f"duong dan: {Path().resolve()}")

duong dan: E:\Tài liệu năm 4\Kỳ 2\datamining\amazon_clothing_project\notebooks


In [2]:
ROOT_DIR      = Path().resolve().parent

# Đường dẫn file RAW
REVIEW_PATH   = ROOT_DIR / "data" / "raw" / "Clothing_Shoes_and_Jewelry.jsonl.gz"
META_PATH     = ROOT_DIR / "data" / "raw" / "meta_Clothing_Shoes_and_Jewelry.jsonl.gz"

# Thư mục output
FIGURES_DIR   = ROOT_DIR / "outputs" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Kiểm tra file có tồn tại không
for name, path in [("Review", REVIEW_PATH), ("Meta", META_PATH)]:
    size_gb = path.stat().st_size / (1024**3) if path.exists() else 0
    status  = f"Tìm được ({size_gb:.2f} GB)" if path.exists() else "không thấy"
    print(f"  {name}: {status}")
    print(f"         Path: {path}")

  Review: Tìm được (6.62 GB)
         Path: E:\Tài liệu năm 4\Kỳ 2\datamining\amazon_clothing_project\data\raw\Clothing_Shoes_and_Jewelry.jsonl.gz
  Meta: Tìm được (3.77 GB)
         Path: E:\Tài liệu năm 4\Kỳ 2\datamining\amazon_clothing_project\data\raw\meta_Clothing_Shoes_and_Jewelry.jsonl.gz


In [7]:
print('xem file review')

with gzip.open(REVIEW_PATH, 'rt', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i >= 3:
            break
        data = json.loads(line)
        print(f"\n--- Dòng {i+1} ---")
        for key, value in data.items():
            # Rút gọn text dài để dễ đọc
            display_val = str(value)[:80] + "..." if len(str(value)) > 80 else value
            print(f"  [{key}]: {display_val}")

print('\n xem file meta')

with gzip.open(META_PATH, 'rt', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i >= 3:
            break
        data = json.loads(line)
        print(f"\n--- Dòng {i+1} ---")
        for key, value in data.items():
            display_val = str(value)[:80] + "..." if len(str(value)) > 80 else value
            print(f"  [{key}]: {display_val}")

xem file review

--- Dòng 1 ---
  [rating]: 3.0
  [title]: Arrived Damaged : liquid in hub locker!
  [text]: Unfortunately Amazon in their wisdom (cough, cough) decided to ship the snowsuit...
  [images]: [{'small_image_url': 'https://m.media-amazon.com/images/I/710WrjJi+hL._SL256_.jp...
  [asin]: B096S6LZV4
  [parent_asin]: B09NSZ5QMF
  [user_id]: AFKZENTNBQ7A7V7UXW5JJI6UGRYQ
  [timestamp]: 1677938767351
  [helpful_vote]: 0
  [verified_purchase]: True

--- Dòng 2 ---
  [rating]: 3.0
  [title]: Useless under 40 degrees.
  [text]: Useless under 40 degrees unless you’re just running to the mailbox & back & don’...
  [images]: []
  [asin]: B09KMDBDCN
  [parent_asin]: B08NGL3X17
  [user_id]: AFKZENTNBQ7A7V7UXW5JJI6UGRYQ
  [timestamp]: 1677083819242
  [helpful_vote]: 0
  [verified_purchase]: False

--- Dòng 3 ---
  [rating]: 4.0
  [title]: Not waterproof, but a very comfy shoe.
  [text]: I purchased these bc they are supposed to be waterproof. Mine are not.  Wore the...
  [images]: []
  [as

In [8]:
def peek_jsonl_gz(filepath, n_rows=5000):
    """Load n_rows đầu tiên từ file .jsonl.gz thành DataFrame"""
    data = []
    with gzip.open(filepath, 'rt', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if i >= n_rows:
                break
            try:
                data.append(json.loads(line.strip()))
            except json.JSONDecodeError:
                continue  # Bỏ qua dòng lỗi
    return pd.DataFrame(data)

df_review_peek = peek_jsonl_gz(REVIEW_PATH, n_rows=5000)
df_meta_peek   = peek_jsonl_gz(META_PATH,   n_rows=5000)

print(f"Review sample shape : {df_review_peek.shape}")
print(f"Meta sample shape   : {df_meta_peek.shape}")

Review sample shape : (5000, 10)
Meta sample shape   : (5000, 16)


 Phân tích cột của Review dataset

In [9]:
print("\nDanh sách cột:")
print(df_review_peek.columns.tolist())

print("\nKiểu dữ liệu & Missing values:")
missing_info = pd.DataFrame({
    'dtype'        : df_review_peek.dtypes,
    'non_null'     : df_review_peek.count(),
    'missing'      : df_review_peek.isnull().sum(),
    'missing_%'    : (df_review_peek.isnull().sum() / len(df_review_peek) * 100).round(2)
})
print(missing_info)
display(df_review_peek.head(3))


Danh sách cột:
['rating', 'title', 'text', 'images', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase']

Kiểu dữ liệu & Missing values:
                     dtype  non_null  missing  missing_%
rating             float64      5000        0        0.0
title               object      5000        0        0.0
text                object      5000        0        0.0
images              object      5000        0        0.0
asin                object      5000        0        0.0
parent_asin         object      5000        0        0.0
user_id             object      5000        0        0.0
timestamp            int64      5000        0        0.0
helpful_vote         int64      5000        0        0.0
verified_purchase     bool      5000        0        0.0


,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,3.0,Arrived Damaged : liquid in hub locker!,"Unfortunately Amazon in their wisdom (cough, c...",[{'small_image_url': 'https://m.media-amazon.c...,B096S6LZV4,B09NSZ5QMF,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1677938767351,0,True
1,3.0,Useless under 40 degrees.,Useless under 40 degrees unless you’re just ru...,[],B09KMDBDCN,B08NGL3X17,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1677083819242,0,False
2,4.0,"Not waterproof, but a very comfy shoe.",I purchased these bc they are supposed to be w...,[],B096N5WK8Q,B07RGM3DYC,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1675524098918,11,True


 Phân tích cột của Meta dataset

In [10]:
print("\nDanh sách cột:")
print(df_meta_peek.columns.tolist())

print("\nKiểu dữ liệu & Missing values:")
missing_info_meta = pd.DataFrame({
    'dtype'     : df_meta_peek.dtypes,
    'non_null'  : df_meta_peek.count(),
    'missing'   : df_meta_peek.isnull().sum(),
    'missing_%' : (df_meta_peek.isnull().sum() / len(df_meta_peek) * 100).round(2)
})
print(missing_info_meta)
display(df_meta_peek.head(3))


Danh sách cột:
['main_category', 'title', 'average_rating', 'rating_number', 'features', 'description', 'price', 'images', 'videos', 'store', 'categories', 'details', 'parent_asin', 'bought_together', 'subtitle', 'author']

Kiểu dữ liệu & Missing values:
                   dtype  non_null  missing  missing_%
main_category     object      4928       72       1.44
title             object      5000        0       0.00
average_rating   float64      5000        0       0.00
rating_number      int64      5000        0       0.00
features          object      5000        0       0.00
description       object      5000        0       0.00
price            float64      2769     2231      44.62
images            object      5000        0       0.00
videos            object      5000        0       0.00
store             object      4984       16       0.32
categories        object      5000        0       0.00
details           object      5000        0       0.00
parent_asin       object     

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author
0,AMAZON FASHION,BALEAF Women's Long Sleeve Zip Beach Coverup U...,4.2,422,"[90% Polyester, 10% Spandex, Zipper closure, M...",[],31.99,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Women's UPF 50+ Front Zip Beach Co...,BALEAF,"[Clothing, Shoes & Jewelry, Women, Clothing, S...","{'Department': 'womens', 'Date First Available...",B09X1MRDN6,None,NaN,NaN
1,AMAZON FASHION,Merrell Work Moab 2 Vent Waterproof SR Boulder,2.7,4,[Rubber sole],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Merrell,"[Clothing, Shoes & Jewelry, Women, Shoes, Outd...",{'Package Dimensions': '14.02 x 9.29 x 4.8 inc...,B073C4Q7W8,None,NaN,NaN
2,AMAZON FASHION,"SAS Women's, Relaxed Sandal",4.7,618,"[Made in the USA, Suede sole, Heel measures ap...","[Unwind, leave your worries behind, and simply...",188.95,[{'thumb': 'https://m.media-amazon.com/images/...,[],SAS,"[Clothing, Shoes & Jewelry, Women, Shoes, Sand...",{'Product Dimensions': '10 x 15 x 6 inches; 2 ...,B0944VG4Y4,None,NaN,NaN
